# Phase 0 — Pretrained detection demo

**Synthetic-to-Real Construction Activity Recognition** — can a model trained
on synthetic excavator video recognise real excavator activity?

This notebook is the first visible commit of the project. It does **not**
train anything yet, and it does **not** yet distinguish activities (digging,
swinging, dumping, travelling, idle) — that is the job of Phase 1 onward. It
runs a *pretrained* YOLO model to prove the whole baseline pipeline works end
to end: install → load model → detect → save an annotated image.

The pretrained model was trained on COCO, so it only knows everyday classes
like `person`, `truck`, `car` — not `excavator` specifically, and it has no
notion of activity at all. Phase 1 trains a dedicated excavator detector on
public construction images (SODA, MOCS); the activity labels (what the
excavator is *doing*) only become possible once the synthetic video pipeline
(Phase 3) and the spatio-temporal models (Phase 5) are in place. For now,
seeing the pipeline detect vehicles and people in a site photo is exactly the
sanity check we want before building any of that.

> **How to use:** in Google Colab, set the runtime to GPU
> (*Runtime → Change runtime type → T4 GPU*), then *Runtime → Run all*.


## 1. Install


In [ ]:
%pip install -q ultralytics


## 2. Load the pretrained model

`yolov8s.pt` is the small model — fast, and more than enough for a demo.
It downloads automatically the first time.


In [ ]:
from ultralytics import YOLO
from PIL import Image

model = YOLO('yolov8s.pt')
print('Model loaded. Classes it knows:', len(model.names))


## 3. Sanity check on a known sample

First we run on Ultralytics' standard sample image (people + a bus). If this
produces boxes, the pipeline is healthy.


In [ ]:
results = model('https://ultralytics.com/images/bus.jpg')
annotated = Image.fromarray(results[0].plot()[..., ::-1])  # BGR -> RGB
annotated.save('sample_output.jpg')
annotated


## 4. Run on a real construction image

Now the interesting part. Upload any public construction-site photo (a site
photo of your own, or one you find online — avoid identifiable faces). The
model will mark every `person`, `truck`, etc. it finds.


In [ ]:
from google.colab import files
uploaded = files.upload()          # pick one image file
fname = list(uploaded.keys())[0]

res = model(fname)
out = Image.fromarray(res[0].plot()[..., ::-1])
out.save('construction_output.jpg')
out


## 5. Why this matters for the real project

The real system does not stop at counting objects in a single frame — it
needs to recognise what the excavator is doing *over time* (digging vs.
swinging vs. idle, say). That needs the video action-recognition models
(SlowFast, X3D, VideoMAE) trained on synthetic clips from Phase 3 onward.

What we can check right now, with only a pretrained detector, is the smallest
building block that every later stage depends on: can we reliably find and
count objects in a single frame at all? Here is that check, applied to the
image above.


In [ ]:
names = res[0].names
counts = {}
for c in res[0].boxes.cls.tolist():
    label = names[int(c)]
    counts[label] = counts.get(label, 0) + 1

print('Detected objects by class:')
for label, n in sorted(counts.items(), key=lambda x: -x[1]):
    print(f'  {label:12s} {n}')


## 6. Commit this

Download `sample_output.jpg` and `construction_output.jpg`, drop them in a
`results/samples/` folder, and commit. That is your first visible commit —
Phase 0 done.

**Next (Phase 1):** download SODA and MOCS, filter/convert the annotations
to YOLO format, and train a dedicated **excavator** detector, then add
ByteTrack and motion rules for an active/idle demo. Ask me for the Phase 1
training notebook when you are ready.
